In [1]:
import pandas as pd
import numpy as np
import json
from utils.tv_var.harmonic_standardizer import HarmonicStandardizer
from utils.tv_var.periodic_var import PeriodicVAR, select_order

import matplotlib.pyplot as plt

#### PCs seleccionadas, 6-horarias, 1940–2026
pcs_sel_6h = pd.read_csv("outputs/pca/pcs_msl_sel_6h.csv", index_col=0, parse_dates=True)

#### Quitar media y desviación estacionales (la estacionalidad va aquí, no en los coeficientes)
harm_std_pcs = HarmonicStandardizer(n_harmonics=3)
harm_std_pcs.fit(pcs_sel_6h)
pcs_sel_6h_z = harm_std_pcs.transform(pcs_sel_6h)
print(pcs_sel_6h_z.shape)

(126864, 30)


In [2]:
bic_0b = select_order(pcs_sel_6h_z, p_values=[6, 8, 10, 12, 16], n_harm=0, freq="6h")
bic_0b

c:\Users\alonsoap\Documents\Proyectos\two_stage_model\utils\tv_var\periodic_var.py:178: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  LAG_MATRIX[f'{col}_lag_{i+1}'] = delay[col].values
c:\Users\alonsoap\Documents\Proyectos\two_stage_model\utils\tv_var\periodic_var.py:178: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  LAG_MATRIX[f'{col}_lag_{i+1}'] = delay[col].values
c:\Users\alonsoap\Documents\Proyectos\two_stage_model\utils\tv_var\periodic_var.py:178: PerformanceWarning: DataFrame is highly fragmented.  This is usually the res

,nobs,loglike,n_params,bic,best
p,,,,,
6,126848,-513297.724746,11010,1.155971e+06,False
8,126848,-487482.521860,12810,1.125492e+06,False
10,126848,-463435.584475,14610,1.098550e+06,True
12,126848,-456562.070906,16410,1.105954e+06,False
16,126848,-440620.490730,20010,1.116373e+06,False


In [3]:
var_h0 = PeriodicVAR(p=10, n_harm=0, freq="6h").fit(pcs_sel_6h_z)
var_h1 = PeriodicVAR(p=10, n_harm=1, freq="6h").fit(pcs_sel_6h_z)
var_h2 = PeriodicVAR(p=10, n_harm=2, freq="6h").fit(pcs_sel_6h_z)

pd.DataFrame({"n_harm=0": [var_h0.n_params(), var_h0.loglike(), var_h0.bic()],
              "n_harm=1": [var_h1.n_params(), var_h1.loglike(), var_h1.bic()],
              "n_harm=2": [var_h2.n_params(), var_h2.loglike(), var_h2.bic()]},
             index=["n_params", "loglike", "bic"])

c:\Users\alonsoap\Documents\Proyectos\two_stage_model\utils\tv_var\periodic_var.py:178: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  LAG_MATRIX[f'{col}_lag_{i+1}'] = delay[col].values
c:\Users\alonsoap\Documents\Proyectos\two_stage_model\utils\tv_var\periodic_var.py:178: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  LAG_MATRIX[f'{col}_lag_{i+1}'] = delay[col].values
c:\Users\alonsoap\Documents\Proyectos\two_stage_model\utils\tv_var\periodic_var.py:178: PerformanceWarning: DataFrame is highly fragmented.  This is usually the res

,n_harm=0,n_harm=1,n_harm=2
n_params,1.461000e+04,3.267000e+04,5.073000e+04
loglike,-4.634874e+05,-3.334518e+05,-3.028441e+05
bic,1.098654e+06,1.050802e+06,1.201806e+06


In [4]:
var_h1.stability()

{'rho': 0.2885797686145233,
 'rho_daily': 0.9966009031037484,
 'rho_step': 0.9991491404488323,
 'stable': True}

In [7]:
print(var_h1.summary())

                         Periodic VAR                         
Variables (k)                   30    msl_PC4, msl_PC1, msl_PC3, msl_PC11, msl_PC6, msl_PC2, msl_PC34, msl_PC8, msl_PC27, msl_PC38, msl_PC17, msl_PC30, msl_PC13, msl_PC42, msl_PC36, msl_PC45, msl_PC12, msl_PC23, msl_PC43, msl_PC41, msl_PC32, msl_PC18, msl_PC35, msl_PC50, msl_PC7, msl_PC20, msl_PC21, msl_PC5, msl_PC24, msl_PC9
Frequency                       6h    0 days 06:00:00
Exogenous (q)                    0    -
Lags (p)                        10
Harmonics (n_harm)               1
Noise                       gaussian / monthly
Sample                    1940-01-03 12:00 → 2026-10-31 18:00
Observations (nobs)         126854
--------------------------------------------------------------
Coefficients                 27090    (903 per equation)
Covariance parameters         5580    (12 block(s))
Total parameters             32670
Log-likelihood                -333451.79
BIC                           1050801.95
------------

#### Simulation

In [8]:
pcs_sim_z = var_h1.simulate(n_years=87, start="1940-01-01", seed=0,
                            burn=int(pd.Timedelta(days=3*365) / pd.Timedelta("6h")))
pcs_sim = harm_std_pcs.inverse_transform(pcs_sim_z)          # vuelve a unidades de PC (con estacionalidad)
print(pcs_sim.shape)

(127107, 30)


In [9]:
import os, pickle, time

out_dir = "outputs/var_6h/simulaciones"
os.makedirs(out_dir, exist_ok=True)

#### Guardar modelo y estandarizador (para el notebook de validación)
with open("outputs/var_6h/var_h1.pkl", "wb") as f:
    pickle.dump(var_h1, f)
with open("outputs/var_6h/harm_std_pcs.pkl", "wb") as f:
    pickle.dump(harm_std_pcs, f)

#### 100 simulaciones de 87 años (misma longitud que 1940–2026)
burn = int(pd.Timedelta(days=3*365) / pd.Timedelta("6h"))
n_sims = 100

for i in range(n_sims):
    f = f"{out_dir}/sim_{i:03d}.parquet"
    #if os.path.exists(f):                                  # ya hecha: se salta
    #    continue
    t = time.time()
    sim_z = var_h1.simulate(n_years=87, start="1940-01-01", seed=i, burn=burn)
    sim = harm_std_pcs.inverse_transform(sim_z).astype("float32")
    sim.to_parquet(f)
    print(f"sim {i:03d}  ({time.time() - t:.0f} s)")

sim 000  (2 s)
sim 001  (2 s)
sim 002  (2 s)
sim 003  (2 s)
sim 004  (2 s)
sim 005  (2 s)
sim 006  (2 s)
sim 007  (2 s)
sim 008  (2 s)
sim 009  (2 s)
sim 010  (2 s)
sim 011  (2 s)
sim 012  (2 s)
sim 013  (2 s)
sim 014  (2 s)
sim 015  (2 s)
sim 016  (2 s)
sim 017  (2 s)
sim 018  (2 s)
sim 019  (2 s)
sim 020  (2 s)
sim 021  (2 s)
sim 022  (2 s)
sim 023  (2 s)
sim 024  (2 s)
sim 025  (2 s)
sim 026  (2 s)
sim 027  (2 s)
sim 028  (2 s)
sim 029  (2 s)
sim 030  (2 s)
sim 031  (2 s)
sim 032  (3 s)
sim 033  (3 s)
sim 034  (2 s)
sim 035  (2 s)
sim 036  (2 s)
sim 037  (2 s)
sim 038  (2 s)
sim 039  (2 s)
sim 040  (2 s)
sim 041  (2 s)
sim 042  (2 s)
sim 043  (2 s)
sim 044  (2 s)
sim 045  (2 s)
sim 046  (2 s)
sim 047  (2 s)
sim 048  (2 s)
sim 049  (2 s)
sim 050  (2 s)
sim 051  (2 s)
sim 052  (2 s)
sim 053  (2 s)
sim 054  (2 s)
sim 055  (2 s)
sim 056  (2 s)
sim 057  (2 s)
sim 058  (2 s)
sim 059  (2 s)
sim 060  (2 s)
sim 061  (2 s)
sim 062  (2 s)
sim 063  (2 s)
sim 064  (2 s)
sim 065  (2 s)
sim 066  (